In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

## Pre-Processing

1.Converting Lower-case

In [8]:
df["review"]=df["review"].str.lower()

2. Removing the URLS

In [9]:
import re

sample_text="abc is th word, abc" #abc=> xyz

new_text=re.sub("abc","xyz",sample_text)

In [10]:
def remove_urls(text):
    re.sub(r"https\s+","",text)
    return text

df["review"]=df["review"].apply(remove_urls)

3.Removing punctuations

In [11]:
def remove_punctuations(text):
    text=re.sub(r"[A-Za-z0-9]","",text)
    return text

df["review"]=df["review"].apply(remove_urls)

4. Removing HTML

In [12]:
def remove_punctuations(text):
    text=re.sub(r"<.*?>","",text)
    return text

df["review"]=df["review"].apply(remove_urls)

5. Removing the Stopwords

In [13]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\korat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\korat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\korat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
#sample_text="I like coding in python!"
#tokens=word_tokenize(sample_text)

In [ ]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    
    return text

df["review"]=df["review"].apply(remove_stopwords)

In [ ]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode 'll h...,positive
1,wderful ltle producti. <br /><br /> filming t...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly 's fmly lttle boy (jke) thks 's zom...,negative
4,"petter mttei's ""love time mey"" vully stun...",positive


6. Stemming

In [ ]:
#Running -> Run
#Played -> Play
#PorterStemming 

from nltk.stem import PorterStemmer

In [ ]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)


df["review"] = df["review"].apply(stemming)

In [ ]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod 'll hook . y rght...,positive
1,wder ltle producti . < br / > < br / > film te...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli 's fmli lttle boy ( jke ) thk 's zomb c...,negative
4,petter mttei 's `` love time mey '' vulli stun...,positive


8 Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

df["sentiment"]=le.fit_transform(df["sentiment"])


In [ ]:
y=df["sentiment"]

In [ ]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

8 Vactorization

In [ ]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod 'll hook . y rght...,1
1,wder ltle producti . < br / > < br / > film te...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli 's fmli lttle boy ( jke ) thk 's zomb c...,0
4,petter mttei 's `` love time mey '' vulli stun...,1


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)

X=tf.fit_transform(df["review"])

Dataset & Data Loaders

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train.shape

(39665, 5000)

In [ ]:
X_test.shape

(9917, 5000)

In [ ]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [ ]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [ ]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader=DataLoader(train_set,batch_size=64,shuffle=True)
test_loader=DataLoader(test_set,batch_size=64,shuffle=True)

#Build the RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connnected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, X):
        # optinal => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, X.size(0), self.hidden_size)

        out, _ = self.rnn(X, h0)
        # 1st value=hidden state of all the timiestamps=> (batch, seq_len, hidden_size)
        # 2nd value=final hidden state of last timestamp

        out = self.fc(out[:, -1, :])
        return out

In [ ]:
input_size= X_train.shape[1]

model=RNN(input_size)

criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

Train the RNN

In [ ]:
epochs=10


for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()

        Xb=Xb.unsqueeze(1)  #add singleton direction 
        
        outputs=model(Xb)  #(batch_size,1)
        
        outputs=torch.sigmoid(outputs.squeeze()) #(batch_size,) =>  probability

        loss=criterion(outputs, yb)  # compute loss
        loss.backward() #backprop
        optimizer.step() #Weights update

    print(f"Epoch: {epoch+1}/{epochs}, Loss: {loss.item()}")

Epoch: 1/10, Loss: 0.27130553126335144
Epoch: 2/10, Loss: 0.43442755937576294
Epoch: 3/10, Loss: 0.26935121417045593
Epoch: 4/10, Loss: 0.28179416060447693
Epoch: 5/10, Loss: 0.2582002580165863
Epoch: 6/10, Loss: 0.33372917771339417
Epoch: 7/10, Loss: 0.3369397222995758
Epoch: 8/10, Loss: 0.1952880322933197
Epoch: 9/10, Loss: 0.3381589651107788
Epoch: 10/10, Loss: 0.22528773546218872


In [ ]:
# evaluate

model.eval()

with torch.no_grad():
    corr_vals = 0
    tot_vals = 0

    for Xb, yb in train_loader:
        Xb = Xb.unsqueeze(1)  # add singleton direction

        outputs = model(Xb)  # (batch_size,1)

        predicted = (
            torch.sigmoid(outputs.squeeze()) > 0.5
        ).float()  # (batch_size,) =>  probability

        tot_vals += yb.size(0)
        corr_vals += (predicted == yb).sum().item()

print(f"Train Accuracy: {corr_vals/tot_vals*100}")

Train Accuracy: 91.95008193621581
